In [1]:
# Install FastAPI and testing libraries
!pip install fastapi pydantic uvicorn httpx

import pandas as pd
import joblib
import os
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from fastapi.testclient import TestClient

# Mount Drive (if Colab disconnected)
from google.colab import drive
drive.mount('/content/drive')

print("Environment ready for API development!")

Mounted at /content/drive
Environment ready for API development!


In [6]:
print("Initializing FastAPI Application...")

# 1. Initialize the App
app = FastAPI(title="Global Fraud Detection Engine", version="1.0")

# 2. Load the Vault (Global state)
models_path = '/content/drive/MyDrive/FraudDetection_project/saved_models/'
print("Loading PaySim brain into server memory...")
xgb_paysim = joblib.load(models_path + 'xgb_paysim_model.joblib')

# 3. Define the Input Schema
class PaySimTransaction(BaseModel):
    amount: float
    balance_drop_ratio: float
    txn_velocity: int
    is_transfer_or_cashout: int
    balance_drained: int
    receiver_balance_unchanged: int

# 4. Create the Health Check Endpoint
@app.get("/")
def health_check():
    return {"status": "Online", "engine": "Fraud Detection API V1"}

# 5. Create the Prediction Endpoint
@app.post("/predict/paysim")
def predict_paysim(transaction: PaySimTransaction):
    try:
        # Convert JSON into a DataFrame using the modern V2 syntax
        input_data = pd.DataFrame([transaction.model_dump()])

        # THE FIX: Force the DataFrame columns to match the exact order the model was trained on
        expected_features = xgb_paysim.feature_names_in_
        input_data = input_data[expected_features]

        # Ask the XGBoost model for the probability of fraud
        fraud_prob = float(xgb_paysim.predict_proba(input_data)[0][1])

        # Business Logic
        is_fraud = bool(fraud_prob >= 0.50)

        if fraud_prob > 0.85:
            risk_tier = "CRITICAL - FREEZE ACCOUNT"
        elif fraud_prob > 0.50:
            risk_tier = "HIGH - BLOCK TRANSACTION"
        elif fraud_prob > 0.20:
            risk_tier = "ELEVATED - REQUIRE 2FA/OTP"
        else:
            risk_tier = "LOW - APPROVE"

        return {
            "transaction_authorized": not is_fraud,
            "fraud_probability": round(fraud_prob, 4),
            "risk_tier": risk_tier,
            "model_version": "xgb_paysim_tuned_v1"
        }

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

print("API Application successfully updated and endpoints registered!")

Initializing FastAPI Application...
Loading PaySim brain into server memory...
API Application successfully updated and endpoints registered!


In [7]:
print("--- RUNNING LIVE API INTEGRATION TESTS ---")

# Spin up a test client to ping our API
client = TestClient(app)

# 1. Ping the Health Check
print("\n[GET /] Pinging Health Check...")
response = client.get("/")
print(f"Status Code: {response.status_code}")
print(f"Response: {response.json()}")

# 2. Simulate a Normal Transaction (e.g., buying a $50 coffee, balance barely drops)
print("\n[POST /predict/paysim] Simulating a SAFE transaction...")
safe_payload = {
    "amount": 50.00,
    "balance_drop_ratio": 0.05,  # Only drained 5% of their account
    "txn_velocity": 1,           # First transaction today
    "is_transfer_or_cashout": 0, # Normal payment
    "balance_drained": 0,        # Account not wiped
    "receiver_balance_unchanged": 0 # Receiver got the money normally
}

response_safe = client.post("/predict/paysim", json=safe_payload)
print(f"Payload sent: {safe_payload}")
print(f"API Decision: {response_safe.json()}")

# 3. Simulate a Hacker Attack (e.g., transferring massive amounts, wiping account, receiver unchanged)
print("\n[POST /predict/paysim] Simulating a FRAUDULENT attack...")
fraud_payload = {
    "amount": 14500.00,
    "balance_drop_ratio": 1.0,   # Drained 100% of their account
    "txn_velocity": 12,          # 12th rapid-fire transaction in 24 hours
    "is_transfer_or_cashout": 1, # High-risk cashout
    "balance_drained": 1,        # Account hit $0.00
    "receiver_balance_unchanged": 1 # Receiver balance magically didn't update (mule account)
}

response_fraud = client.post("/predict/paysim", json=fraud_payload)
print(f"Payload sent: {fraud_payload}")
print(f"API Decision: {response_fraud.json()}")

--- RUNNING LIVE API INTEGRATION TESTS ---

[GET /] Pinging Health Check...
Status Code: 200
Response: {'status': 'Online', 'engine': 'Fraud Detection API V1'}

[POST /predict/paysim] Simulating a SAFE transaction...
Payload sent: {'amount': 50.0, 'balance_drop_ratio': 0.05, 'txn_velocity': 1, 'is_transfer_or_cashout': 0, 'balance_drained': 0, 'receiver_balance_unchanged': 0}
API Decision: {'transaction_authorized': True, 'fraud_probability': 0.0016, 'risk_tier': 'LOW - APPROVE', 'model_version': 'xgb_paysim_tuned_v1'}

[POST /predict/paysim] Simulating a FRAUDULENT attack...
Payload sent: {'amount': 14500.0, 'balance_drop_ratio': 1.0, 'txn_velocity': 12, 'is_transfer_or_cashout': 1, 'balance_drained': 1, 'receiver_balance_unchanged': 1}
API Decision: {'transaction_authorized': False, 'fraud_probability': 0.9999, 'risk_tier': 'CRITICAL - FREEZE ACCOUNT', 'model_version': 'xgb_paysim_tuned_v1'}
